<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/solved/08_lognormal_distributional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 8 — Distributional lognormal model

Model not only the expected reaction-time trajectory but also participant-to-participant differences in residual variability.

## Setup

This course pins PyMC, modular ArviZ, and Bambi for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1" \
    "bambi==0.21.0"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bambi as bmb
import pymc as pm
import arviz_base as azb
import arviz_stats as azs
import arviz_plots as azp

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("Bambi:", bmb.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every Bambi model in this sequence uses `center_predictors=False`. The `Intercept` prior is therefore a prior on baseline reaction time rather than reaction time at the average deprivation day.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

# 8.1 Modeling residual heterogeneity

Do participants differ in residual variability as well as in their mean reaction-time trajectories?

Bambi distributional formulas let another likelihood parameter have its own regression. Here `sigma` has a log link and a participant-specific intercept:

$$\log(\sigma_i)=\gamma_0+v_{s[i]}.$$

The mean trajectory still has varying intercepts and slopes. The chapter's joint LKJ correlation structure across varying parameters is not available in Bambi, so these hierarchical components are independent.

In [ ]:
formula = bmb.Formula(
    "Reaction ~ Days + (1 + Days | Subject)",
    "sigma ~ 1 + (1 | Subject)",
)

priors = {
    "mu": {
        "Intercept": bmb.Prior("Normal", mu=5.5, sigma=0.55),
        "Days": bmb.Prior("Normal", mu=0, sigma=0.20),
        "1|Subject": bmb.Prior(
            "Normal", mu=0, sigma=bmb.Prior("Exponential", lam=3)
        ),
        "Days|Subject": bmb.Prior(
            "Normal", mu=0, sigma=bmb.Prior("Exponential", lam=5)
        ),
    },
    "sigma": {
        "Intercept": bmb.Prior("Normal", mu=0, sigma=0.30),
        "1|Subject": bmb.Prior(
            "Normal", mu=0, sigma=bmb.Prior("Exponential", lam=3)
        ),
    },
}

model = bmb.Model(
    formula, sleep, family="lognormal", priors=priors, categorical="Subject",
    center_predictors=False,
)
model

In [ ]:
prior = model.prior_predictive(draws=500, random_seed=RANDOM_SEED)
azp.plot_ppc_dist(
    prior,
    group="prior_predictive",
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 8.2 Participant-specific residual scales

How large are the participant-specific residual scales, and how much do they vary?

In [ ]:
idata = model.fit(draws=1000, tune=2000, chains=4, target_accept=0.95, random_seed=RANDOM_SEED)
print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))


In [ ]:
azs.summary(
    idata,
    var_names=["Intercept", "Days", "1|Subject_sigma", "Days|Subject_sigma", "sigma_Intercept", "sigma_1|Subject_sigma"],
    ci_prob=0.90, ci_kind="hdi", round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["Intercept", "Days", "1|Subject_sigma", "Days|Subject_sigma", "sigma_Intercept", "sigma_1|Subject_sigma"],
);

In [ ]:
bmb.interpret.plot_predictions(
    model,
    idata,
    conditional="Days",
    average_by="Subject",
    target="Reaction",
    prob=[0.50, 0.90],
)

In [ ]:
bmb.interpret.plot_predictions(
    model,
    idata,
    conditional=["Days", "Subject"],
    target="Reaction",
    prob=0.90,
    subplot_kwargs={"main": "Days", "panel": "Subject"},
    fig_kwargs={"wrap": 6},
)

In [ ]:
bmb.interpret.plot_predictions(
    model,
    idata,
    conditional="Subject",
    target="sigma",
    prob=0.90,
)

# 8.3 Predictive value of heteroskedasticity

Does modeling participant-specific residual variability improve the predictive description of reaction times?

In [ ]:
model.predict(
    idata,
    kind="response",
    inplace=True,
    random_seed=RANDOM_SEED,
)

azp.plot_ppc_dist(
    idata,
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 8.4 Eliciting scale hierarchies

What makes the residual-scale hierarchy difficult to specify scientifically?

Distributional models can describe heteroskedasticity, but their scale parameters live on transformed scales and are harder to elicit. The chapter next explores an ex-Gaussian likelihood, which represents right-skew directly as a Gaussian component plus an exponential tail.